In [0]:
# --- Paths ---
landing_path = "abfss://bronze@stretailsalesdeveus001.dfs.core.windows.net/landing/sales/"
checkpoint_path = "/Volumes/retail_sales_dev/bronze/checkpoints/sales/checkpoint"
schema_location = "/Volumes/retail_sales_dev/bronze/checkpoints/sales/schema"

# --- Read stream using Auto Loader ---
raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "CustomerID STRING, Invoice STRING")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(landing_path)
)
    

In [0]:
# display(raw_df.limit(20))
from pyspark.sql.functions import current_timestamp, col

raw_df_with_meta = (
    raw_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

In [0]:
query = (
    raw_df_with_meta.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("retail_sales_dev.bronze.sales_raw")
)

query.awaitTermination()

In [0]:
%sql

-- SELECT * FROM retail_sales_dev.bronze.sales_raw LIMIT 20;

SELECT COUNT(*) FROM retail_sales_dev.bronze.sales_raw;


In [0]:
%sql

SELECT CustomerID, InvoiceNo, InvoiceDate, _source_file, _ingested_at
FROM retail_sales_dev.bronze.sales_raw
LIMIT 20;

DESCRIBE TABLE retail_sales_dev.bronze.sales_raw;

In [0]:
%sql
   SELECT _source_file, COUNT(*) 
   FROM retail_sales_dev.bronze.sales_raw 
   GROUP BY _source_file;

In [0]:
# dbutils.fs.rm('/Volumes/retail_sales_dev/bronze/checkpoints/sales/schema/', recurse=True)

In [0]:
# %sql
raw_df.printSchema()